# Qwen PE — 実モデルで文章書き換え
サイトで保存した設定を読み込み、実モデルを実行して結果JSONをダウンロードします。

「ランタイム → ランタイムのタイプを変更」でCUDA GPUを選んでください。無料T4での動作は未保証です。余裕のあるGPUメモリ・通常RAM・ディスクが必要です。

この開発環境ではGPU・有効キーでの実行は未検証です。失敗時はエラーで停止し、模擬結果を生成しません。依存関係は公式開発版を含み、環境によって調整が必要です。

Colabは対話的な実験用です。このノートブックは公開URL・トンネルを作りません。実行後はランタイムを停止してください。
[PE-T2I](https://huggingface.co/Qwen/Qwen-Image-2.1-PE-T2I) / [PE-I2I](https://huggingface.co/Qwen/Qwen-Image-2.1-PE-I2I)。編集モードでは実画像も入力します。生成するのは文章であり、画像生成モデルはここでは起動しません。


## 1. インストール
パッケージを更新した後にランタイム再起動を求められた場合は、再起動し、次のセルから進んでください。

In [ ]:
%pip install "torch>=2.4.0" "transformers>=5.17" accelerate pillow huggingface_hub


## 2. 設定を読み込む
サイトから保存した設定JSONを1つ選びます。参照画像は後の実行セルで選びます。

In [ ]:
"""Shared helpers embedded into the Colab notebooks. No public server or tunnel."""
import base64
import datetime
import hashlib
import io
import json
import math
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

KINDS = ('comfyui', 'inference', 'prompt-rewrite', 'evals')

def validate_config(c, expected):
    if c.get('schema') != 'atlas-colab-config-v1' or c.get('kind') != expected:
        raise ValueError('別のデモの設定ファイルです。対象サイトから保存し直してください。')
    if not isinstance(c.get('prompt'), str) or not 1 <= len(c['prompt'].strip()) <= 4500:
        raise ValueError('プロンプトを確認してください。')
    for key, minimum, maximum in [('seed', 0, 2147483647), ('steps', 1, 50), ('repetitions', 1, 20)]:
        if type(c.get(key)) is not int or not minimum <= c[key] <= maximum:
            raise ValueError('設定値が不正です: ' + key)
    if c.get('size') not in (1024, 2048) or c.get('mode') not in ('generate', 'edit'):
        raise ValueError('サイズ・モードが不正です。')
    if not isinstance(c.get('transparent'), bool):
        raise ValueError('透過設定が不正です。')
    return c

def load_config(expected):
    from google.colab import files
    print('サイトから保存した atlas-' + expected + '-config.json を選択してください。')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('設定JSONは1件だけ選択してください。')
    raw = next(iter(uploaded.values()))
    if len(raw) > 20000:
        raise ValueError('設定ファイルが大きすぎます。')
    return validate_config(json.loads(raw), expected)

def gpu_info(required=True):
    try:
        result = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        if required:
            raise RuntimeError('CUDA GPUランタイムを選択してください。')
        result = 'CPU (API evaluation)'
    return {'gpu': result, 'python': platform.python_version()}

def effective_prompt(c):
    return ('This is an RGBA image with transparency. ' + c['prompt'] + ' The image has alpha channel and the background is transparent.') if c['transparent'] else c['prompt']

def upload_images(maximum=10):
    from google.colab import files
    from PIL import Image
    uploaded = files.upload()
    if not 1 <= len(uploaded) <= maximum:
        raise ValueError(f'画像は1〜{maximum}枚です。')
    images = []
    total = 0
    for data in uploaded.values():
        total += len(data)
        if len(data) > 4 * 1024**2 or total > 20 * 1024**2:
            raise ValueError('画像は1枚4MB、合計20MBまでです。')
        img = Image.open(io.BytesIO(data))
        if img.format not in ('PNG', 'JPEG', 'WEBP') or img.width * img.height > 20_000_000:
            raise ValueError('PNG/JPEG/WebP、2000万画素以下を選択してください。')
        img.load()
        images.append(img)
    return images

def image_data(image):
    output = io.BytesIO()
    image.save(output, format='PNG')
    if len(output.getvalue()) > 16 * 1024**2:
        raise ValueError('結果が16MBを超えました。1024pxで再実行してください。')
    return 'data:image/png;base64,' + base64.b64encode(output.getvalue()).decode('ascii')

def report(c, engine, model, elapsed, env, **extra):
    return {'schema': 'atlas-colab-result-v1', 'kind': c['kind'],
            'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'engine': engine, 'model': model, 'elapsed_seconds': elapsed,
            'environment': env, 'config': c, **extra}

def save_report(result):
    from google.colab import files
    path = Path('/content/atlas-' + result['kind'] + '-' + result['engine'] + '-result.json')
    path.write_text(json.dumps(result, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    files.download(str(path))
    print('結果JSONをサイトに読み込んでください。APIキーは結果に含めません。')

config=load_config('prompt-rewrite')
print(json.dumps(config,ensure_ascii=False,indent=2))
print(gpu_info(True))


## 3. 実モデルを実行
初回はモデルのダウンロードに時間がかかります。表示される画像・スコアは実行したモデルの結果です。

In [ ]:
import torch
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModelForImageTextToText

def run_rewrite(c):
    env = gpu_info()
    if not torch.cuda.is_available(): raise RuntimeError('CUDA GPUを選択してください。')
    model_id = 'Qwen/Qwen-Image-2.1-PE-' + ('I2I' if c['mode']=='edit' else 'T2I')
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    system = Path(hf_hub_download(model_id, 'system_prompt.txt')).read_text()
    prompt = effective_prompt(c)
    if c['mode'] == 'edit':
        refs = [image.convert('RGB') for image in upload_images()]
        processor = AutoProcessor.from_pretrained(model_id)
        model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, device_map='auto').eval()
        messages = [{'role':'system','content':[{'type':'text','text':system}]}, {'role':'user','content':[{'type':'image','image':image} for image in refs]+[{'type':'text','text':prompt}]}]
        inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt', enable_thinking=True).to(model.device)
        tokenizer = processor.tokenizer
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, device_map='auto').eval()
        text = tokenizer.apply_chat_template([{'role':'system','content':system},{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
    torch.manual_seed(c['seed'])
    torch.cuda.synchronize(); start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=24000 if c['mode']=='edit' else 16256, do_sample=True, temperature=1.0, top_p=.95, top_k=20)
    torch.cuda.synchronize(); elapsed = time.perf_counter()-start
    decoded = tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    answer = decoded.rsplit('</think>',1)[-1].strip()
    if answer.startswith('```'):
        answer = '\n'.join(answer.splitlines()[1:-1])
    result = json.loads(answer)
    if not isinstance(result.get('rewritten_prompt'),str) or not result['rewritten_prompt'].strip():
        raise ValueError('モデルが有効な書き換え文を返しませんでした。')
    print(result['rewritten_prompt'])
    env.update(transformers=__import__('transformers').__version__, torch=torch.__version__, dtype=str(dtype))
    return report(c, 'qwen-pe', model_id, elapsed, env, rewrite=result, timing_scope='generation_only', system_prompt_sha256=hashlib.sha256(system.encode()).hexdigest())

result=run_rewrite(config)


## 4. 結果を保存してサイトへ戻る
結果JSONには入力文や生成物が含まれます。対象のデモで「結果JSONを読み込む」を選びます。サイトの読み込みはブラウザ内だけで処理します。

In [ ]:
save_report(result)
